# Demo: Sistema RAG para EcoMarket con Llama 3.1 8B
Este notebook demuestra paso a paso el funcionamiento del sistema RAG del Taller 2.

In [1]:
# Instalación (si es necesario)
# !pip install langchain-community chromadb ollama

## 1. Importar librerías y verificar Ollama

In [2]:
import subprocess
import sys

# Verificar que Ollama esté corriendo
try:
    subprocess.run(["ollama", "--version"], capture_output=True, check=True)
    print("✅ Ollama instalado")
except:
    print("❌ Ollama no encontrado. Instálalo desde https://ollama.com")
    sys.exit(1)

✅ Ollama instalado


In [3]:
import os
from langchain_community.llms import Ollama
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

## 2. Base de conocimiento

In [4]:
documento = """Nuestra política de devoluciones permite a los clientes devolver productos dentro de los 30 días posteriores a la fecha de compra. 
Para ser elegible para una devolución, el artículo debe estar sin usar, en su embalaje original y en las mismas condiciones en que lo recibió. 
Los productos perecederos, como alimentos y flores, no son elegibles para devolución. 
Las devoluciones de ropa de segunda mano también están sujetas a una revisión estricta para garantizar que el artículo no haya sido usado después de la compra. 
Para iniciar una devolución, visite nuestra sección de ayuda en el sitio web y siga los pasos."""

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
docs = splitter.create_documents([documento])
print(f"Fragmentos generados: {len(docs)}")

Fragmentos generados: 5


## 3. Crear embeddings y vectorstore con Chroma

In [6]:
embeddings = OllamaEmbeddings(model="llama3.1:8b")
vectorstore = Chroma.from_documents(docs, embedding=embeddings)
print("Vectorstore lista.")

Vectorstore lista.


## 4. Configurar LLM y cadena RAG

In [7]:
llm = Ollama(model="llama3.1:8b", temperature=0.1)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

template = """Usa solo el siguiente contexto para responder:
{context}
Pregunta: {question}
Respuesta:"""
prompt = PromptTemplate(template=template, input_variables=["context", "question"])

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt}
)
print("Cadena RAG creada.")

Cadena RAG creada.


## 5. Ejecutar preguntas de ejemplo

In [8]:
pregunta = "¿Cuántos días tengo para devolver un producto?"
respuesta = qa_chain.run(pregunta)
print(f"Pregunta: {pregunta}\nRespuesta: {respuesta}")

c:\Users\14624165\GitHub\IAGenerativa\.venv312\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(


Pregunta: ¿Cuántos días tengo para devolver un producto?
Respuesta: Tienes 30 días.


In [9]:
pregunta2 = "¿Aceptan devolución de alimentos?"
respuesta2 = qa_chain.run(pregunta2)
print(f"Pregunta: {pregunta2}\nRespuesta: {respuesta2}")

Pregunta: ¿Aceptan devolución de alimentos?
Respuesta: Lo siento, pero no hay información en el contexto proporcionado que indique si aceptan devolución de alimentos. La política de devoluciones mencionada se refiere a productos en general y no especifica qué tipo de artículos están cubiertos por esta política. Si necesitas más detalles o una respuesta específica sobre la devolución de alimentos, te recomendaría contactar directamente con el servicio de atención al cliente del proveedor para obtener información precisa.


## 6. Conclusión
El sistema RAG responde correctamente usando solo el contexto proporcionado, evitando alucinaciones.